# Repair suite: Gaussian denoiser, DPAR и structured corruption

Notebook запускается после успешной sentiment baseline validation. Frozen prompts, seeds, judge, steering vector и intervention layer не меняются.

Проверяются: Gaussian denoiser из задания; DPAR как защита steering direction; structured corruption; norm-preserving и `lambda=0.5` ablations. Validation steering direction никогда не используется при обучении denoiser.


In [ ]:
import os, subprocess, pathlib
repo = pathlib.Path('/content/steering-manifold-repair')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/Nek1tt/steering-manifold-repair.git', str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'], check=True)
os.chdir(repo)
print('cwd:', os.getcwd())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.'], check=True)
subprocess.run([sys.executable,'-m','pytest','-q'], check=True)

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Switch runtime to GPU before training.'


## 0. Проверка frozen baseline
Repair suite ожидает `results/sentiment_direction.pt`. В чистом runtime воспроизводится тот же валидированный direction; это не новый поиск baseline.


In [ ]:
from pathlib import Path
if not Path('results/sentiment_direction.pt').exists():
    subprocess.run([sys.executable,'scripts/validate_sentiment_baseline.py','--config','configs/baseline_sentiment_gpt2.yaml'], check=True)
else:
    print('Using existing frozen direction:', Path('results/sentiment_direction.pt'))

## 1. Кэш generic natural activations
Используется WikiText-2 только как generic LM text. Кэшируется до 80k `blocks.6.hook_resid_post` activation vectors; sentiment evaluation data в обучение не попадает.


In [ ]:
subprocess.run([sys.executable,'scripts/cache_activations.py','--config','configs/repair_suite_gpt2.yaml'], check=True)

## 2. Обучить Gaussian denoiser
Corruption magnitude нормируется относительно `||h||`; residual MLP начинает близко к identity.


In [ ]:
subprocess.run([sys.executable,'scripts/train_denoiser.py','--config','configs/repair_suite_gpt2.yaml','--kind','gaussian'], check=True)

## 3. Обучить mixed structured denoiser
Половина non-identity corruptions использует нормированное natural activation difference `h_j-h_k`, половина — Gaussian noise.


In [ ]:
subprocess.run([sys.executable,'scripts/train_denoiser.py','--config','configs/repair_suite_gpt2.yaml','--kind','mixed'], check=True)

## 3b. Cross-corruption reconstruction
Оба checkpoints оцениваются на одинаковых held-out Gaussian и structured corruptions, чтобы отделить reconstruction specialization от downstream steering behavior.


In [ ]:
subprocess.run([sys.executable,'scripts/eval_denoiser_reconstruction.py','--config','configs/repair_suite_gpt2.yaml'], check=True)

## 4. Оценить все repair hypotheses
Методы: additive, norm-preserving, Gaussian, Gaussian `lambda=.5`, Gaussian DPAR, mixed, mixed DPAR. Основной grid: `alpha={0,.5,.75,1,1.5,2,3,4}`.


In [ ]:
subprocess.run([sys.executable,'scripts/eval_repairs.py','--config','configs/repair_suite_gpt2.yaml'], check=True)

## 5. Pareto и mechanistic diagnostics
Кроме final text metrics измеряем requested/effective `alpha`, cosine correction со steering vector, parallel fraction и correction/steering norm. Это проверяет, не улучшает ли vanilla denoiser fluency простой отменой steering.


In [ ]:
subprocess.run([sys.executable,'scripts/plot_repairs.py','--config','configs/repair_suite_gpt2.yaml'], check=True)

In [ ]:
from IPython.display import display, Image, Markdown
for name in ['repair_pareto.png','effective_alpha.png','correction_geometry.png']:
    p = Path('results/repair_suite')/name
    if p.exists(): display(Image(filename=str(p)))
report = Path('results/repair_suite/hypothesis_report.md')
if report.exists(): display(Markdown(report.read_text()))

In [ ]:
import pandas as pd
agg = pd.read_csv('results/repair_suite/repair_aggregate.csv')
frontier = pd.read_csv('results/repair_suite/frontier_summary.csv')
display(frontier)
cols = ['method','strength','fluency_score','concept_score','effective_alpha','alpha_preservation_error','correction_cos_v','correction_parallel_fraction']
display(agg[cols].sort_values(['method','strength']))

## Как интерпретировать
Главное — не требовать победы каждой гипотезы. Если vanilla repair уменьшает effective `alpha`, это steering cancellation. Если DPAR сохраняет requested `alpha`, геометрическая гипотеза подтверждается. Structured corruption считается полезной только при downstream improvement, а не только при reconstruction. Frozen baseline после просмотра repair results не подстраивается.
